[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CathalD/NorthStarProject_CoastalBlueCarbonMMRV/blob/main/CoastalBlueCarbon_GlobalCoreCovariate_Extraction.ipynb)

# Global Soil Core Covariate Extraction — Coastal Blue Carbon
## Harmonized WOSIS + CanPeat + Janousek → GEE Remote Sensing Covariates

Extracts remote sensing covariate bands at all harmonized soil profile locations from
`combined_profiles_filtered.csv` (output of `05_filter_profiles.R`) using the
**Google Earth Engine Python API** (`reduceRegions`).

### Canonical 27-band covariate stack (v1.7/v1.8)
Covariate names and order are **identical** across all three scripts — required for
transfer learning (`P4_05_Transfer_Learning_Modelling.R`):

| Group | Bands | Count |
|-------|-------|-------|
| Topography & Channels | elevation_m, slope, elevationRelMHW, twi, **dist_to_channel_m**, tidal_flat_prob, coastal_dist_m | 7 |
| Sentinel-1 SAR | VV_mean, VH_mean, VVVH_ratio | 3 |
| Sentinel-2 Optical & Phenology | B, G, R, **B5, B6, B7**, NIR, SWIR1, SWIR2, NDVI_median, LSWI_median, mNDWI_median, **NDVI_stdDev**, SAVI_median, tidal_wetness | 15 |
| Climate | MAT_C, MAP_mm | 2 |
| **Total** | | **27** |

> **Changes from previous version:** `tpi` → `dist_to_channel_m` (more direct hydrological driver);
> Red-Edge bands B5/B6/B7 added (chlorophyll proxies for seagrass/saltmarsh);
> `NDVI_stdDev` added (phenological variability); `EVI_median` removed (redundant with Red-Edge).
>
> **SoilGrids:** Used as external Bayesian prior only — NOT a canonical covariate band.
> `stack_sg` is defined in Cell 3 for reference, but is not extracted at core locations.
> Uncertainty inflation applied in R: `P3_0c_bayesian_prior_setup_bluecarbon.R`.
>
> **Ecosystem scope:** Tidal marsh (EM) and seagrass (SG) only — VM0033 eligible.

### Two-stage filtering
| Stage | Where | Filter |
|-------|-------|--------|
| **Geographic + depth** | R script 05 | lat ≥ 30°N, lon -170 to +60°E, depth ≥ 30 cm |
| **Climate analog** | This notebook (Cells 3c–5b) | MAT within ±2°C and MAP within ±30% of AOI centroid |

> **Efficiency note:** TerraClimate extracted first (Cell 5), climate filter applied (Cell 5b),
> reducing the point set before the expensive Sentinel-2 extraction (Cell 5d).

### Cluster-then-Match embedding similarity (Cell 6)
K-means clusters AOI pixels → extracts medoid vectors → cosine similarity at each core
→ `embedding_max_sim` used as per-core transfer-learning weight in R.

### Outputs
| File | Content |
|------|---------|
| `CorePoints_Covariates_BC_Canada.csv` | 27 canonical bands at climate-filtered points |
| `CorePoints_EmbeddingSimilarity_BC_Canada.csv` | Cosine similarity to k=5 AOI clusters |
| `AOI_Medoids.csv` | k × 64 medoid embedding vectors |
| `AOI_KMeans_Clusters.tif` | 2-band GeoTIFF: cluster label + cosine similarity |
| `CorePoints_Climate_Filtered_OUT.csv` | Audit — profiles removed by climate filter |

---
### Prerequisites
1. Run `05_filter_profiles.R` → `combined_profiles_filtered.csv`
2. Upload CSV (Cell 2)
3. Run cells in order: Auth → Load → Stacks → Climate ref → TerraClimate → Filter → Rebuild → Extract → Cluster → Export


In [ ]:
# ============================================================================
# 1. AUTHENTICATION + SETUP
# ============================================================================
import ee

# Authenticate with your Google account (browser popup)
ee.Authenticate()

# Initialise with the project that holds your GEE assets
ee.Initialize(project='north-star-project-470316')

import pandas as pd
import numpy as np
from google.colab import files

print('✓ GEE authenticated and initialised')

In [ ]:
# ============================================================================
# 2. LOAD GEOGRAPHICALLY + DEPTH FILTERED PROFILES → BUILD GEE FEATURE LIST
# ============================================================================
# Input: combined_profiles_filtered.csv (output of 05_filter_profiles.R)
#   Filters already applied: lat >= 30°N, lon -170 to +60°E, depth >= 30 cm
# Climate analog filtering (MAT/MAP) is applied in Cell 5b of this notebook.

# Option A (Colab): upload the file directly
print('Upload combined_profiles_filtered.csv when prompted...')
uploaded = files.upload()
FILENAME = list(uploaded.keys())[0]

# Option B (Drive): uncomment and mount Google Drive instead
# from google.colab import drive
# drive.mount('/content/drive')
# FILENAME = '/content/drive/MyDrive/SoilCarbonProfiles/combined_profiles_filtered.csv'

df = pd.read_csv(FILENAME)
print(f'Loaded: {len(df):,} profiles (post geographic + depth filter)')
print(f'Datasets: {df["dataset"].value_counts().to_dict()}')
print(f'Columns: {list(df.columns)}')

# Build one ee.Feature per profile
# Only profile_id + dataset stored as GEE properties; all other attributes
# are merged back in pandas after extraction.
df_pts = df.dropna(subset=['latitude', 'longitude']).copy()
n_missing = len(df) - len(df_pts)
if n_missing > 0:
    print(f'⚠ {n_missing} rows dropped — missing latitude or longitude')

feature_list = []
for _, row in df_pts.iterrows():
    geom = ee.Geometry.Point([float(row['longitude']), float(row['latitude'])])
    feature_list.append(ee.Feature(geom, {
        'profile_id': str(row['profile_id']),
        'dataset':    str(row['dataset'])
    }))

print(f'✓ {len(feature_list):,} GEE features created (all geo+depth filtered profiles)')

In [ ]:
# ============================================================================
# 3. DEFINE REMOTE SENSING IMAGE STACKS
# ============================================================================
# Canonical 27-band stack — MUST match GoogleEarthEngineAOICovariateAnalysis.js
# and CoastalBlueCarbon_LargeScaleCovariateExtraction.ipynb exactly.
#
# Group 1 — Topography & Channels (7):
#   elevation_m, slope, elevationRelMHW, twi, dist_to_channel_m,
#   tidal_flat_prob, coastal_dist_m
# Group 2 — Sentinel-1 SAR (3):
#   VV_mean, VH_mean, VVVH_ratio
# Group 3 — Sentinel-2 Optical & Phenology (15):
#   B, G, R, B5, B6, B7, NIR, SWIR1, SWIR2
#   NDVI_median, LSWI_median, mNDWI_median, NDVI_stdDev, SAVI_median, tidal_wetness
# Group 4 — Climate (2): MAT_C, MAP_mm  [extracted separately in Cell 3b]
#
# NOTE on SoilGrids:
#   stack_sg is defined below for use as a Bayesian prior in the R workflow
#   (P3_0c_bayesian_prior_setup_bluecarbon.R). It is NOT part of the canonical
#   27-band covariate stack and is NOT extracted at core point locations.
#   Red-Edge bands (B5, B6, B7) replace the three former SoilGrids covariate
#   slots — they are primary chlorophyll-sensitive predictors for Zostera
#   (eelgrass) and Spartina (saltmarsh) carbon burial rates.
# ============================================================================

# ── DATE RANGES (match GEE JS script) ────────────────────────────
S2_START  = '2020-01-01'
S2_END    = '2023-12-31'
SAR_START = '2020-01-01'
SAR_END   = '2023-12-31'

# Sentinel-2 extraction parameters
S2_MAX_CLOUD   = 20    # % cloud cover threshold for initial scene filter (matches JS S2_CLOUD_THRESHOLD)
S2_IMAGE_LIMIT = 20    # max scenes kept per batch area (least cloudy first)
S2_BUFFER_M    = 5000  # metres buffer around each batch of points


# ── GROUP 1: Topography & Channels ───────────────────────────────
dem         = ee.Image('NASA/NASADEM_HGT/001').select('elevation')
elevation_m = dem.rename('elevation_m')
slope       = ee.Terrain.slope(dem).rename('slope')

# Elevation relative to Mean High Water (MHW)
# Approximation: DEM - 0.5 m proxy for MHW above MSL.
# For site-specific work, replace 0.5 with your local MHW offset.
elevRelMHW = dem.subtract(0.5).rename('elevationRelMHW')

# TWI — Topographic Wetness Index = ln(upslope_area / tan(slope))
# High TWI = flat/concave terrain prone to waterlogging (marsh hollows, channels)
slope_rad = ee.Terrain.slope(dem).multiply(3.14159 / 180)
tan_slope = slope_rad.tan().max(0.001)  # guard against log(0)
contrib   = (dem.gte(-9999).unmask(0)
             .reduceNeighborhood(reducer=ee.Reducer.sum(),
                                 kernel=ee.Kernel.circle(**{'radius': 20, 'units': 'pixels'}))
             .max(1))
twi = contrib.divide(tan_slope).log().rename('twi')

# dist_to_channel_m — distance to tidal channels & permanent waterways (replaces TPI v1.4)
# Uses JRC Global Surface Water occurrence > 30% as the channel mask.
# Lower threshold (30%) captures ephemeral tidal channels in addition to permanent water.
# focalMax(30 m) closes canopy-gap artefacts before the distance transform.
# fastDistanceTransform returns squared-euclidean pixel distance;
# sqrt() + multiply by 30 converts to metres at 30 m scale.
channel_mask = (ee.Image('JRC/GSW1_4/GlobalSurfaceWater')
                .select('occurrence').gt(30).unmask(0)
                .focalMax(30, 'circle', 'meters'))
dist_to_channel = (
    channel_mask
    .fastDistanceTransform(500, 'pixels', 'squared_euclidean')
    .sqrt()
    .multiply(30)   # pixels → metres at 30 m native scale
    .rename('dist_to_channel_m').float()
)

# Tidal flat probability — Murray et al. 2019 Global Intertidal Change
# Asset: UQ/murray/Intertidal/v1_1/global_intertidal (ImageCollection, NOT a single Image)
# FIX v1.8: Use ee.ImageCollection(...).filterBounds().mosaic() — avoids asset-type error
# and scopes the compositor to the local region to prevent global-scale timeout.
try:
    tidal_flat_prob = (
        ee.ImageCollection('UQ/murray/Intertidal/v1_1/global_intertidal')
        .filterBounds(ee.Geometry.BBox(-180, -90, 180, 90))  # global fallback; narrow in batch fn
        .mosaic()
        .select('classification').eq(1).unmask(0)
        .rename('tidal_flat_prob').float()
    )
    print('✓ Murray intertidal dataset loaded (ImageCollection mosaic)')
except Exception:
    tidal_flat_prob = ee.Image(0).rename('tidal_flat_prob').float()
    print('⚠ Murray intertidal dataset unavailable — tidal_flat_prob set to 0')

# Coastal distance — derived from JRC Global Surface Water occurrence > 50%
# fastDistanceTransform returns pixel distance; × 30 to convert to metres
water_mask   = ee.Image('JRC/GSW1_4/GlobalSurfaceWater').select('occurrence').gt(50).unmask(0)
coastal_dist = (
    water_mask.fastDistanceTransform(500, 'pixels', 'squared_euclidean')
    .sqrt().multiply(30).rename('coastal_dist_m').float()
)

stack_topo = (elevation_m.addBands(slope).addBands(elevRelMHW)
              .addBands(twi).addBands(dist_to_channel)
              .addBands(tidal_flat_prob).addBands(coastal_dist))
print('✓ Group 1 — Topography & Channels (7):')
print('    elevation_m, slope, elevationRelMHW, twi, dist_to_channel_m,')
print('    tidal_flat_prob, coastal_dist_m')
print('    (tpi removed; dist_to_channel_m added — more direct hydrological driver)')


# ── GROUP 2: Sentinel-1 SAR ───────────────────────────────────────
# FIX v1.6: Noise filter applied per-pixel via .map() — not as a collection
# metadata property filter (which caused the collection to return empty).
s1_col = (
    ee.ImageCollection('COPERNICUS/S1_GRD')
    .filterDate(SAR_START, SAR_END)
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
    .filter(ee.Filter.eq('instrumentMode', 'IW'))
    .map(lambda img: img.updateMask(img.select('VV').gt(-30)))   # pixel-level noise mask
)
s1_mean = s1_col.mean()
vv      = s1_mean.select('VV').rename('VV_mean')
vh      = s1_mean.select('VH').rename('VH_mean')
vv_vh   = s1_mean.select('VV').subtract(s1_mean.select('VH')).rename('VVVH_ratio')
stack_s1 = vv.addBands(vh).addBands(vv_vh)
print('✓ Group 2 — SAR (3): VV_mean, VH_mean, VVVH_ratio')


# ── GROUP 3: Sentinel-2 Optical & Phenology ───────────────────────
# Per-batch spatial filter approach (see extract_batch_s2 below).
# Summer months (May–Sep): peak tidal marsh biomass in BC.
#
# CHANGES vs previous version:
#   - EVI_median REMOVED (replaced by NDVI_stdDev and Red-Edge bands)
#   - B5, B6, B7 (Red-Edge) ADDED — primary chlorophyll proxies for seagrass/saltmarsh
#   - NDVI_stdDev ADDED — phenological variability; distinguishes seasonal tidal marsh
#     from permanently-green coastal shrubs (must be computed BEFORE median)
#   - NDBI removed (previous version); brightness + greenness TC removed (previous version)

def process_s2_image(image):
    """Cloud mask + tidal inundation mask + scale to reflectance."""
    qa         = image.select('QA60')
    cloud_mask = qa.bitwiseAnd(1 << 10).eq(0).And(qa.bitwiseAnd(1 << 11).eq(0))
    ndwi_check = image.normalizedDifference(['B3', 'B8'])
    tide_mask  = ndwi_check.lt(0.1)   # keep only exposed pixels (NDWI < 0.1)
    return image.updateMask(cloud_mask.And(tide_mask)).divide(10000)

def make_s2_median_for_region(region_geom):
    """Build spatially filtered S2 median for a specific region (per-batch).
    Includes B5, B6, B7 (Red-Edge) alongside standard bands."""
    col = (
        ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterDate(S2_START, S2_END)
        .filterBounds(region_geom)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', S2_MAX_CLOUD))
        .filter(ee.Filter.calendarRange(5, 9, 'month'))   # May–September
        .limit(S2_IMAGE_LIMIT, 'CLOUDY_PIXEL_PERCENTAGE')
        .map(process_s2_image)
    )
    return col.median().clip(region_geom)

# NDVI_stdDev — computed across the full summer collection BEFORE taking the median.
# High = seasonally dynamic vegetation (tidal marsh). Low = permanent water or
# stable vegetation. Must be a static image extracted via extract_batch(), not S2.
# FIX v1.7: .select('NDVI_stdDev') restricts the variance pass to one band,
# reducing memory pressure during global-scale reduction.
_ndvi_col_for_std = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterDate(S2_START, S2_END)
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', S2_MAX_CLOUD))
    .filter(ee.Filter.calendarRange(5, 9, 'month'))
    .map(process_s2_image)
    .map(lambda img: img.normalizedDifference(['B8', 'B4']).rename('NDVI_stdDev'))
)
stack_ndvi_stddev = _ndvi_col_for_std.reduce(ee.Reducer.stdDev()).rename('NDVI_stdDev')
print('✓ Group 3 — Sentinel-2 Optical & Phenology (15): per-batch spatial filter (May–Sep, NDWI mask)')
print('    Raw (9): B, G, R, B5, B6, B7, NIR, SWIR1, SWIR2')
print('    Derived (6): NDVI_median, LSWI_median, mNDWI_median, NDVI_stdDev, SAVI_median, tidal_wetness')
print('    (EVI removed; B5/B6/B7 Red-Edge added; NDVI_stdDev added as phenology metric)')


# ── SoilGrids SOC — BAYESIAN PRIOR ONLY (not a canonical covariate band) ──
# stack_sg is kept here for reference and for the Bayesian R prior workflow.
# It is NOT extracted at core point locations and NOT in CANONICAL_BANDS.
# SoilGrids is trained on upland soils — high uncertainty in tidal sediments.
# Uncertainty inflation (×1.2–1.5) applied in: P3_0c_bayesian_prior_setup_bluecarbon.R
# Correct band names on soc_mean: soc_0-5cm_mean, soc_5-15cm_mean, etc.
# (Previous version incorrectly used 'ocs_0-5cm_mean' — that band does not exist)
sg      = ee.Image('projects/soilgrids-isric/soc_mean')
sg_bdod = ee.Image('projects/soilgrids-isric/bdod_mean')
def _ocs_layer(soc_band, bd_band, thickness):
    """OCS (kg/m²) = SOC (dg/kg)/10 × BD (cg/cm³)/100 × thickness(cm)/100"""
    return (sg.select(soc_band).divide(10)
            .multiply(sg_bdod.select(bd_band).divide(100))
            .multiply(thickness).divide(100))
ocs_0_5    = _ocs_layer('soc_0-5cm_mean',   'bdod_0-5cm_mean',    5).rename('sg_ocs_0_5cm')
ocs_5_15   = _ocs_layer('soc_5-15cm_mean',  'bdod_5-15cm_mean',  10).rename('sg_ocs_5_15cm')
ocs_15_30  = _ocs_layer('soc_15-30cm_mean', 'bdod_15-30cm_mean', 15).rename('sg_ocs_15_30cm')
ocs_30_60  = _ocs_layer('soc_30-60cm_mean', 'bdod_30-60cm_mean', 30).rename('sg_ocs_30_60cm')
ocs_60_100 = _ocs_layer('soc_60-100cm_mean','bdod_60-100cm_mean',40).rename('sg_ocs_60_100cm')
sg_0_30   = ocs_0_5.add(ocs_5_15).add(ocs_15_30).rename('sg_soc_0_30cm')
sg_30_100 = ocs_30_60.add(ocs_60_100).rename('sg_soc_30_100cm')
sg_0_100  = sg_0_30.add(sg_30_100).rename('sg_soc_0_100cm')
stack_sg  = sg_0_30.addBands(sg_30_100).addBands(sg_0_100)
print('✓ SoilGrids SOC prior defined (Bayesian R workflow only — NOT a canonical covariate):')
print('    sg_soc_0_30cm, sg_soc_30_100cm, sg_soc_0_100cm')


# ── GROUP 6: Google Satellite Embedding V1 (64 bands) ─────────────
# Unchanged — used for cluster-then-match transfer learning weights in Cell 6.
stack_embed = ee.ImageCollection('GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL').median()
n_embed_bands = stack_embed.bandNames().size().getInfo()
print(f'✓ Group 6 — Google Satellite Embedding V1: {n_embed_bands} bands at 10 m')
print()
print('═══════════════════════════════════════════════════════')
print('Canonical 27-band stack: 7 topo/channels + 3 SAR + 15 S2 + 2 climate = 27')
print('Plus 64-band embedding (separate extraction for cluster-then-match)')
print('SoilGrids: external Bayesian prior only (NOT in canonical stack)')


In [ ]:
# ============================================================================
# 3b. DEFINE TERRACLIMATE STACK (MAT + MAP)
# Consistent with charlies_place_v4_2.js: IDAHO_EPSCOR/TERRACLIMATE, 2000-2022
# Units in raw GEE data:
#   tmmn / tmmx : °C × 10  (divide by 10 to get °C)
#   pr          : mm/month  (multiply monthly mean × 12 to get mm/year)
# ============================================================================

TC_START = '2000-01-01'
TC_END   = '2022-12-31'

terra_col = (
    ee.ImageCollection('IDAHO_EPSCOR/TERRACLIMATE')
    .filterDate(TC_START, TC_END)
    .select(['tmmn', 'tmmx', 'pr'])
)
terra_mean = terra_col.mean()   # long-term monthly mean

# Mean Annual Temperature (°C)
# MAT = ((tmmn_mean + tmmx_mean) / 2) / 10
mat_img = terra_mean.expression(
    '((tmmn + tmmx) / 2.0) / 10.0',
    {'tmmn': terra_mean.select('tmmn'),
     'tmmx': terra_mean.select('tmmx')}
).rename('MAT_C')

# Mean Annual Precipitation (mm/year)
# mean monthly pr × 12 months
map_img = terra_mean.select('pr').multiply(12).rename('MAP_mm')

stack_climate = mat_img.addBands(map_img)
print(f'✓ Stack Climate — TerraClimate {TC_START[:4]}–{TC_END[:4]}: MAT_C (°C), MAP_mm (mm/yr)')

In [ ]:
# ============================================================================
# 3c. SAMPLE TERRACLIMATE AT CHARLIE'S PLACE KBA CENTROID
#     → Set AOI reference MAT and MAP for climate analog filtering
# ============================================================================

# Climate analog tolerances
MAT_TOLERANCE = 2.0   # ±2°C
MAP_TOLERANCE = 0.30  # ±30%

# Load AOI from GEE asset and get its centroid
AOI_ASSET = 'projects/blue-carbon-hub/assets/Charlies_Place_KBA_boundaryFile_2025'
aoi_fc     = ee.FeatureCollection(AOI_ASSET)
aoi_geom   = aoi_fc.geometry()
aoi_centroid = aoi_geom.centroid(maxError=100)

# Sample TerraClimate at the AOI centroid
# TerraClimate native resolution ~4 km (scale=4000 m)
aoi_sample = stack_climate.sample(
    region=aoi_centroid,
    scale=4000,
    numPixels=1,
    geometries=False
).first().getInfo()

AOI_MAT = aoi_sample['properties']['MAT_C']
AOI_MAP = aoi_sample['properties']['MAP_mm']

# Compute filter bounds
MAT_MIN = AOI_MAT - MAT_TOLERANCE
MAT_MAX = AOI_MAT + MAT_TOLERANCE
MAP_MIN = AOI_MAP * (1.0 - MAP_TOLERANCE)
MAP_MAX = AOI_MAP * (1.0 + MAP_TOLERANCE)

print(f"Charlie's Place KBA — reference climate (TerraClimate {TC_START[:4]}–{TC_END[:4]})")
print(f'  MAT = {AOI_MAT:.2f} °C')
print(f'  MAP = {AOI_MAP:.0f} mm/year')
print(f'')
print(f'Climate analog filter ranges:')
print(f'  MAT: {MAT_MIN:.1f} to {MAT_MAX:.1f} °C   (±{MAT_TOLERANCE} °C)')
print(f'  MAP: {MAP_MIN:.0f} to {MAP_MAX:.0f} mm/year  (±{MAP_TOLERANCE*100:.0f}%)')

In [ ]:
# ============================================================================
# 4. BATCH EXTRACTION FUNCTIONS
# ============================================================================
# Two functions:
#
#   extract_batch()     — general purpose: used for Topo, SAR, TerraClimate,
#                         NDVI_stdDev, and Embedding. Accepts a pre-built ee.Image.
#                         clip_buffer_m=0 means no clipping (global layers).
#
#   extract_batch_s2()  — Sentinel-2 specific: rebuilds a spatially filtered
#                         S2 median from scratch for each batch's bounding box.
#                         This is the only reliable way to avoid GEE computing
#                         a full global mosaic per call, which causes timeouts
#                         even with batch_size=8 and tileScale=4.
#
# Both use reduceRegions() in WGS84, avoiding the UTM Zone projection error
# that broke the GEE JS sampleRegions() call for global Janousek points.
# ============================================================================

def extract_batch(image, features, name, batch_size=100, scale=30, clip_buffer_m=0):
    """Extract pixel values at point locations using batched reduceRegions."""
    results   = []
    n_batches = (len(features) + batch_size - 1) // batch_size
    n_failed  = 0
    print(f'--- Extracting {name} ({len(features):,} pts, batch={batch_size}, scale={scale}m) ---')

    for i in range(0, len(features), batch_size):
        batch     = features[i: i + batch_size]
        fc        = ee.FeatureCollection(batch)
        batch_num = i // batch_size + 1
        try:
            img = image.clip(fc.geometry().buffer(clip_buffer_m)) if clip_buffer_m > 0 else image
            res = img.reduceRegions(
                collection = fc,
                reducer    = ee.Reducer.first(),
                scale      = scale,
                tileScale  = 2
            )
            data = res.getInfo()['features']
            for f in data:
                results.append(f['properties'])
            if batch_num % 10 == 0 or batch_num == n_batches:
                print(f'   Batch {batch_num}/{n_batches} OK  ({len(results):,} rows so far)')
        except Exception as e:
            n_failed += 1
            print(f'   Batch {batch_num}/{n_batches} FAILED: {e}')

    print(f'✓ {name} complete — {len(results):,} rows, {n_failed} batches failed')
    return pd.DataFrame(results)


def extract_batch_s2(features, name, band_selector_fn,
                     batch_size=25, scale=30, buffer_m=None):
    """
    Sentinel-2 specific extraction.

    For each batch:
      1. Compute a bounding box + buffer around the batch points (server-side)
      2. Call make_s2_median_for_region() to build a spatially filtered median
         using only the ~20 least-cloudy S2 scenes covering that area
      3. Apply band_selector_fn to pick raw or derived bands
      4. reduceRegions() on just those points

    This ensures GEE never touches more S2 imagery than needed for
    the small geographic footprint of each batch.

    band_selector_fn : callable(s2_median_image) → ee.Image with renamed bands
    buffer_m         : spatial buffer around batch bbox (defaults to S2_BUFFER_M)
    """
    if buffer_m is None:
        buffer_m = S2_BUFFER_M

    results   = []
    n_batches = (len(features) + batch_size - 1) // batch_size
    n_failed  = 0
    print(f'--- Extracting {name} ({len(features):,} pts, batch={batch_size}, '
          f'scale={scale}m, buffer={buffer_m}m) ---')

    for i in range(0, len(features), batch_size):
        batch     = features[i: i + batch_size]
        fc        = ee.FeatureCollection(batch)
        batch_num = i // batch_size + 1
        try:
            # Build a buffered region around this batch
            region = fc.geometry().bounds().buffer(buffer_m)

            # Build a spatially filtered S2 median for this region only
            s2_local = make_s2_median_for_region(region)

            # Select / compute the desired bands
            img = band_selector_fn(s2_local)

            res = img.reduceRegions(
                collection = fc,
                reducer    = ee.Reducer.first(),
                scale      = scale,
                tileScale  = 2
            )
            data = res.getInfo()['features']
            for f in data:
                results.append(f['properties'])
            if batch_num % 10 == 0 or batch_num == n_batches:
                print(f'   Batch {batch_num}/{n_batches} OK  ({len(results):,} rows so far)')
        except Exception as e:
            n_failed += 1
            print(f'   Batch {batch_num}/{n_batches} FAILED: {e}')

    print(f'✓ {name} complete — {len(results):,} rows, {n_failed} batches failed')
    return pd.DataFrame(results)


# ── Band selector functions for S2 ───────────────────────────────
def s2_raw_bands(s2):
    """
    Select 9 raw reflectance bands from a per-batch S2 median image.
    Includes Red-Edge bands B5 (705 nm), B6 (740 nm), B7 (783 nm).
    These are primary chlorophyll-sensitive predictors for Zostera (eelgrass)
    and Spartina (saltmarsh) carbon burial rates.
    Band order matches the JS script canonical sequence: B, G, R, B5, B6, B7, NIR, SWIR1, SWIR2.
    """
    return (
        s2.select('B2').rename('B')
        .addBands(s2.select('B3').rename('G'))
        .addBands(s2.select('B4').rename('R'))
        .addBands(s2.select('B5').rename('B5'))   # Red-Edge 705 nm
        .addBands(s2.select('B6').rename('B6'))   # Red-Edge 740 nm
        .addBands(s2.select('B7').rename('B7'))   # Red-Edge 783 nm
        .addBands(s2.select('B8').rename('NIR'))
        .addBands(s2.select('B11').rename('SWIR1'))
        .addBands(s2.select('B12').rename('SWIR2'))
    )

def s2_derived_bands(s2):
    """
    Compute 5 derived indices + Tasseled Cap Wetness from a per-batch S2 median.

    CHANGES vs previous version:
      - EVI_median REMOVED — replaced by NDVI_stdDev (phenology) and Red-Edge bands.
        EVI is redundant when B5/B6/B7 chlorophyll proxies are available.
      - NDBI REMOVED (previous version) — irrelevant to tidal marsh / seagrass.
      - Tasseled Cap brightness + greenness REMOVED (previous version) — forest metrics.
      - tidal_wetness RETAINED (Nedkov 2017 S2 SR coefficients) — most inundation-sensitive
        TC component for coastal ecosystems.
      - NDVI_stdDev is NOT computed here — it requires a collection, not a single median
        image. See stack_ndvi_stddev in cell-stacks, extracted via extract_batch().
    """
    B     = s2.select('B2')
    G     = s2.select('B3')
    R     = s2.select('B4')
    NIR   = s2.select('B8')
    SWIR1 = s2.select('B11')
    SWIR2 = s2.select('B12')

    # Vegetation / biomass indices
    ndvi  = s2.normalizedDifference(['B8', 'B4']).rename('NDVI_median')
    savi  = s2.expression(
        '((NIR - RED) / (NIR + RED + 0.5)) * 1.5',
        {'NIR': NIR, 'RED': R}
    ).rename('SAVI_median')

    # Water / inundation indices (key for tidal marsh & seagrass)
    lswi  = s2.normalizedDifference(['B8', 'B11']).rename('LSWI_median')
    mndwi = s2.normalizedDifference(['B3', 'B11']).rename('mNDWI_median')

    # Tasseled Cap Wetness — Nedkov 2017 Sentinel-2 SR coefficients.
    # More positive = wetter / flooded / high soil moisture.
    tidal_wetness = s2.expression(
        '0.1511*B + 0.1973*G + 0.3283*R + 0.3407*NIR + (-0.7117)*SWIR1 + (-0.4559)*SWIR2',
        {'B': B, 'G': G, 'R': R, 'NIR': NIR, 'SWIR1': SWIR1, 'SWIR2': SWIR2}
    ).rename('tidal_wetness')

    return (
        ndvi.addBands(lswi).addBands(mndwi)
        .addBands(savi)
        .addBands(tidal_wetness)
    )

print('✓ extract_batch() and extract_batch_s2() defined')
print('✓ s2_raw_bands() updated: 9 bands (B, G, R, B5, B6, B7, NIR, SWIR1, SWIR2)')
print('✓ s2_derived_bands() updated: 5 bands (NDVI_median, LSWI_median, mNDWI_median, SAVI_median, tidal_wetness)')
print('    EVI_median removed; NDVI_stdDev extracted separately (see stack_ndvi_stddev)')


In [ ]:
# ============================================================================
# 5. EXTRACT TERRACLIMATE (MAT + MAP) AT ALL GEO+DEPTH FILTERED POINTS
# Extracted FIRST (before Topo/S1/S2) at ~4 km resolution — cheap and fast.
# Climate filter is applied immediately after (Cell 5b) to reduce the point
# set before the expensive Sentinel-2 extraction runs (Cell 5d).
# ============================================================================
df_climate = extract_batch(stack_climate, feature_list, 'TerraClimate (MAT/MAP)',
                           batch_size=500, scale=4000, clip_buffer_m=0)

print(f'\nTerraClimate extraction complete: {len(df_climate):,} rows')
if 'MAT_C' in df_climate.columns:
    print('\nMAT_C (°C) distribution:')
    print(df_climate['MAT_C'].describe().round(2))
    print('\nMAP_mm distribution:')
    print(df_climate['MAP_mm'].describe().round(0))

In [ ]:
# ============================================================================
# 5b. APPLY CLIMATE ANALOG FILTER
#     Keep profiles whose MAT and MAP are within tolerance of AOI
# ============================================================================
def merge_df(main, sub, key='profile_id'):
    """Left-join extracted values onto the main profiles dataframe."""
    if sub is None or sub.empty:
        print('  ⚠ Skipping empty extraction result')
        return main
    drop_cols = ['system:index', 'dataset']
    keep_cols = [key] + [c for c in sub.columns if c not in [key] + drop_cols]
    return pd.merge(main, sub[keep_cols], on=key, how='left')


# Merge climate values onto profile dataframe
df_climate['profile_id'] = df_climate['profile_id'].astype(str)
df_with_climate = df_pts.copy()
df_with_climate['profile_id'] = df_with_climate['profile_id'].astype(str)
df_with_climate = merge_df(df_with_climate, df_climate)

n_has_climate = df_with_climate['MAT_C'].notna().sum() if 'MAT_C' in df_with_climate.columns else 0
print(f'Profiles with TerraClimate data: {n_has_climate:,} / {len(df_with_climate):,}')

# Climate filter:
#   - Keep if MAT AND MAP are within bounds
#   - Also keep if climate data is missing (NaN) — don't exclude unknowns
if 'MAT_C' in df_with_climate.columns:
    within_mat = (df_with_climate['MAT_C'] >= MAT_MIN) & (df_with_climate['MAT_C'] <= MAT_MAX)
    within_map = (df_with_climate['MAP_mm'] >= MAP_MIN) & (df_with_climate['MAP_mm'] <= MAP_MAX)
    climate_pass = within_mat & within_map
    no_data      = df_with_climate['MAT_C'].isna()
    keep_mask    = climate_pass | no_data
else:
    print('⚠ MAT_C column not found — skipping climate filter (check TerraClimate extraction)')
    keep_mask = pd.Series([True] * len(df_with_climate), index=df_with_climate.index)

df_filtered = df_with_climate[keep_mask].copy()
df_removed  = df_with_climate[~keep_mask].copy()

print(f'\n── Climate filter summary ──────────────────────────────────')
print(f'  Input (post geo+depth filter):  {len(df_with_climate):,} profiles')
print(f'  AOI MAT = {AOI_MAT:.2f} °C  →  keep {MAT_MIN:.1f}–{MAT_MAX:.1f} °C')
print(f'  AOI MAP = {AOI_MAP:.0f} mm  →  keep {MAP_MIN:.0f}–{MAP_MAX:.0f} mm/yr')
print(f'  Removed (outside climate range): {len(df_removed):,}')
print(f'  Retained:                        {len(df_filtered):,}')
print(f'\nRemoved by dataset:')
print(df_removed['dataset'].value_counts().to_string() if len(df_removed) > 0 else '  (none)')

# Save audit file of removed profiles
AUDIT_FILE = 'CorePoints_Climate_Filtered_OUT.csv'
df_removed.to_csv(AUDIT_FILE, index=False)
files.download(AUDIT_FILE)
print(f'\n✓ Audit file saved: {AUDIT_FILE}  ({len(df_removed):,} removed profiles)')

In [ ]:
# ============================================================================
# 5c. REBUILD FEATURE LIST FROM CLIMATE-FILTERED PROFILES
#     All subsequent extractions use this filtered list
# ============================================================================
feature_list_filtered = []
for _, row in df_filtered.iterrows():
    geom = ee.Geometry.Point([float(row['longitude']), float(row['latitude'])])
    feature_list_filtered.append(ee.Feature(geom, {
        'profile_id': str(row['profile_id']),
        'dataset':    str(row['dataset'])
    }))

print(f'✓ Climate-filtered feature list: {len(feature_list_filtered):,} points')
print(f'  (Reduced from {len(feature_list):,} geo+depth filtered points)')
print(f'  Breakdown: {df_filtered["dataset"].value_counts().to_dict()}')

In [ ]:
# ============================================================================
# 5d. EXTRACT REMOTE SENSING STACKS AT CLIMATE-FILTERED POINTS
# ============================================================================
# Extraction order:
#   Stack 1 — Topography & Channels (7 bands, 30 m): static global DEM layers
#   Stack 2 — Sentinel-1 SAR (3 bands, 30 m): global mean with pixel noise mask
#   Stack 3a — Sentinel-2 Raw (9 bands, 30 m): per-batch median (B, G, R, B5, B6, B7, NIR, SWIR1, SWIR2)
#   Stack 3b — Sentinel-2 Derived (5 bands, 30 m): per-batch median
#              (NDVI_median, LSWI_median, mNDWI_median, SAVI_median, tidal_wetness)
#   Stack 3c — NDVI_stdDev (1 band, 30 m): pre-computed seasonal phenology image
#              extracted via extract_batch() — not from the per-batch S2 median
#
# Note: TerraClimate (Group 4, 2 bands) was already extracted in Cell 5
# and its values are already merged into df_filtered from Cell 5b.
#
# Note: Google Embedding V1 (64 bands) is extracted separately in Cell 6
# for the Cluster-then-Match analysis.
#
# Note: SoilGrids is NOT extracted here — it is an external Bayesian prior
# only. See stack_sg definition in cell-stacks for R workflow reference.
# ============================================================================

# Stack 1: Topography & Channels — static global DEM + JRC, no clipping needed
df_topo = extract_batch(stack_topo, feature_list_filtered, 'Topography & Channels',
                        batch_size=500, scale=30, clip_buffer_m=0)

# Stack 2: Sentinel-1 SAR — global mean with pixel noise mask, no clip needed
df_s1   = extract_batch(stack_s1, feature_list_filtered, 'Sentinel-1 SAR',
                        batch_size=100, scale=30, clip_buffer_m=0)

# Stack 3a: Sentinel-2 raw reflectance (9 bands: B, G, R, B5, B6, B7, NIR, SWIR1, SWIR2)
# extract_batch_s2() filters the collection to the batch region internally,
# so only ~20 local scenes are ever processed per batch.
df_s2_raw = extract_batch_s2(
    feature_list_filtered, 'Sentinel-2 Raw (9 bands incl. Red-Edge)', s2_raw_bands,
    batch_size=25, scale=30
)

# Stack 3b: Sentinel-2 derived indices (5 bands: NDVI, LSWI, mNDWI, SAVI, tidal_wetness)
# EVI_median removed; replaced by Red-Edge bands and NDVI_stdDev.
# Separate call — same per-batch approach, independent S2 median built each time.
df_s2_derived = extract_batch_s2(
    feature_list_filtered, 'Sentinel-2 Derived (5 bands)', s2_derived_bands,
    batch_size=25, scale=30
)

# Stack 3c: NDVI_stdDev — seasonal phenology variability, pre-computed before median.
# Must be extracted as a static image (not per-batch S2), because it is computed
# over the full summer collection rather than a single median composite.
# batch_size=100 is safe at 30 m global scale; increase if timeouts occur.
df_ndvi_std = extract_batch(stack_ndvi_stddev, feature_list_filtered, 'NDVI StdDev (phenology)',
                             batch_size=100, scale=30, clip_buffer_m=0)

print('\n=== Remote sensing extraction complete ===')
print(f'  Topo & Channels (7 bands): {len(df_topo):,} rows | {len(df_topo.columns)} cols')
print(f'  Sentinel-1 (3 bands):      {len(df_s1):,} rows | {len(df_s1.columns)} cols')
print(f'  S2 Raw (9 bands):          {len(df_s2_raw):,} rows | {len(df_s2_raw.columns)} cols')
print(f'  S2 Derived (5 bands):      {len(df_s2_derived):,} rows | {len(df_s2_derived.columns)} cols')
print(f'  NDVI StdDev (1 band):      {len(df_ndvi_std):,} rows | {len(df_ndvi_std.columns)} cols')
print(f'  TerraClimate (2 bands):    already in df_filtered (extracted in Cell 5)')
print(f'  SoilGrids:                 NOT extracted — external Bayesian prior only')


In [ ]:
# ============================================================================
# 6. CLUSTER-THEN-MATCH — AOI K-MEANS + EMBEDDING SIMILARITY
# ============================================================================
# Strategy
# --------
# Instead of exporting all 64 raw embedding bands, we:
#   A. Sample all AOI pixels at 30 m → run K-means (GEE server-side, k=5)
#   B. Extract the medoid vector for each cluster (pixel closest to mean)
#      → downloads only k × 64 values from GEE
#   C. Extract 64-band embedding at each climate-filtered core location
#   D. Compute cosine similarity (Python/numpy) — cores vs. medoids
#      → N_cores × k similarity scores used as transfer learning weights
#   E. Build cluster-label + similarity raster image for GeoTIFF export
#
# Outputs (built here, exported in Cell 7):
#   sim_df          — profile_id + embedding_sim_to_cluster_0..4 +
#                     embedding_max_sim + embedding_best_cluster
#   aoi_output_img  — 2-band GEE image (cluster_id, sim_to_nearest_medoid)
#   medoids_data    — raw getInfo() result (k rows × 64 bands)
# ============================================================================

import numpy as np

# ── USER-TUNABLE CONFIGURATION ────────────────────────────────────
N_CLUSTERS     = 5    # k for K-means on AOI pixels
AOI_GRID_SCALE = 30   # metres — pixel spacing for AOI pixel sampling
EMBED_SCALE    = 10   # metres — native resolution of Google Embedding V1

print(f'Cluster-then-Match config: k={N_CLUSTERS}, AOI grid={AOI_GRID_SCALE}m, embed={EMBED_SCALE}m')

# ═══════════════════════════════════════════════════════════════════
# STEP A: Sample AOI pixels + run K-means (GEE server-side)
# ═══════════════════════════════════════════════════════════════════
print('\n── Step A: Sampling AOI pixels at 30 m and running K-means ──')

# Sample all pixels inside the AOI on a regular 30 m grid.
# geometries=True keeps coordinates so we can export the cluster image.
aoi_pixels = stack_embed.sample(
    region     = aoi_geom,
    scale      = AOI_GRID_SCALE,
    seed       = 42,
    geometries = True
)

n_aoi_pixels = aoi_pixels.size().getInfo()
print(f'  AOI pixels sampled: {n_aoi_pixels:,}')

# Train K-means clusterer on the sampled AOI pixels (server-side)
clusterer   = ee.Clusterer.wekaKMeans(N_CLUSTERS, seed=42).train(aoi_pixels)

# Apply cluster labels back to the sampled pixels (FeatureCollection)
aoi_clustered = aoi_pixels.cluster(clusterer)   # adds 'cluster' property to each Feature

# Apply cluster labels as a raster for GeoTIFF export
aoi_cluster_img = stack_embed.cluster(clusterer).rename('cluster_id')

print(f'  K-means complete: {N_CLUSTERS} clusters assigned to AOI pixels')

# ═══════════════════════════════════════════════════════════════════
# STEP B: Extract medoid vector per cluster
#   Medoid = the actual AOI pixel whose embedding is closest to the
#   cluster mean vector. Downloads only k × 64 values from GEE.
# ═══════════════════════════════════════════════════════════════════
print(f'\n── Step B: Extracting medoid (nearest pixel to mean) for each cluster ──')

# Get the full list of embedding band names once
embed_band_names = stack_embed.bandNames().getInfo()

def get_cluster_medoid(cluster_id):
    """Return the AOI pixel (as ee.Feature) closest to its cluster mean."""
    cluster_pts = aoi_clustered.filter(ee.Filter.eq('cluster', cluster_id))

    # Compute mean embedding for this cluster
    cluster_mean_dict = cluster_pts.reduceColumns(
        reducer   = ee.Reducer.mean().repeat(len(embed_band_names)),
        selectors = embed_band_names
    ).get('mean')

    mean_img = ee.Image.constant(cluster_mean_dict)

    # Squared Euclidean distance from each pixel to the cluster mean
    def add_dist(feature):
        pt_img    = ee.Image.constant(
            ee.List(embed_band_names).map(lambda b: feature.get(b))
        )
        sq_dist   = pt_img.subtract(mean_img).pow(2).reduce(ee.Reducer.sum())
        dist_val  = sq_dist.reduceRegion(
            reducer  = ee.Reducer.first(),
            geometry = feature.geometry(),
            scale    = EMBED_SCALE
        ).get('sum')
        return feature.set('dist_to_mean', dist_val)

    ranked = cluster_pts.map(add_dist).sort('dist_to_mean')
    medoid = ranked.first()
    return medoid.set('cluster_id', cluster_id)

medoids_fc   = ee.FeatureCollection(
    [get_cluster_medoid(i) for i in range(N_CLUSTERS)]
)
medoids_data = medoids_fc.getInfo()   # download: k rows × 64 bands

print(f'  Medoids downloaded: {len(medoids_data["features"])} cluster representatives')
for feat in medoids_data['features']:
    cid = feat['properties'].get('cluster_id', '?')
    n_bands = sum(1 for k in feat['properties'] if k in embed_band_names)
    print(f'    Cluster {cid}: {n_bands} embedding bands')

# ═══════════════════════════════════════════════════════════════════
# STEP C: Extract 64-band embedding at each climate-filtered core
# ═══════════════════════════════════════════════════════════════════
print(f'\n── Step C: Extracting 64-band embedding at {len(feature_list_filtered):,} core locations ──')

df_embed = extract_batch(stack_embed, feature_list_filtered, 'Google Embedding V1',
                         batch_size=50, scale=EMBED_SCALE, clip_buffer_m=0)

print(f'  Core embeddings: {len(df_embed):,} rows | {len(df_embed.columns)} cols')
embed_cols = [c for c in df_embed.columns
              if c not in ['profile_id', 'dataset', 'system:index']]
print(f'  Embedding bands ({len(embed_cols)}): {embed_cols[:4]} … {embed_cols[-2:]}')

# ═══════════════════════════════════════════════════════════════════
# STEP D: Compute cosine similarity (Python/numpy)
#   cosine_sim(core_i, medoid_j) = dot(norm(core_i), norm(medoid_j))
# ═══════════════════════════════════════════════════════════════════
print(f'\n── Step D: Computing cosine similarity ({len(df_embed):,} cores × {N_CLUSTERS} medoids) ──')

# Core embedding matrix  — N_cores × 64
core_matrix = df_embed[embed_cols].values.astype(float)

# Medoid embedding matrix — N_CLUSTERS × 64
medoid_matrix = np.array([
    [feat['properties'].get(b, np.nan) for b in embed_cols]
    for feat in medoids_data['features']
], dtype=float)

def l2_norm(mat):
    """Row-wise L2 normalisation; zero-vectors left at zero (avoid /0)."""
    norms = np.linalg.norm(mat, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return mat / norms

core_norm   = l2_norm(np.nan_to_num(core_matrix,   nan=0.0))
medoid_norm = l2_norm(np.nan_to_num(medoid_matrix, nan=0.0))

# Cosine similarity: N_cores × N_CLUSTERS
sim_matrix = core_norm @ medoid_norm.T

# Build output dataframe
sim_df = df_embed[['profile_id']].copy()
for j in range(N_CLUSTERS):
    sim_df[f'embedding_sim_to_cluster_{j}'] = sim_matrix[:, j]
sim_df['embedding_max_sim']      = sim_matrix.max(axis=1)
sim_df['embedding_best_cluster'] = sim_matrix.argmax(axis=1)

print(f'  sim_df shape: {sim_df.shape}')
print(f'  embedding_max_sim summary:')
print(sim_df['embedding_max_sim'].describe().round(4).to_string())
print(f'\n  Best-cluster distribution:')
print(sim_df['embedding_best_cluster'].value_counts().sort_index().to_string())

# ═══════════════════════════════════════════════════════════════════
# STEP E: Build per-pixel cluster-similarity raster (for GeoTIFF)
#   Band 1: cluster_id (0–k-1)
#   Band 2: cosine similarity of each pixel to its cluster's medoid
# ═══════════════════════════════════════════════════════════════════
print(f'\n── Step E: Building cluster + similarity raster over AOI ──')

def cosine_sim_image(medoid_feature, band_names):
    """
    Return a single-band GEE image of cosine similarity between
    stack_embed and one medoid vector, over the full AOI extent.
    """
    props      = medoid_feature['properties']
    medoid_vec = [float(props.get(b, 0.0)) for b in band_names]

    medoid_img = ee.Image.constant(medoid_vec).rename(band_names)

    # dot product (numerator)
    dot = stack_embed.multiply(medoid_img).reduce(ee.Reducer.sum())

    # L2 norm of embedding at each pixel (denominator term 1)
    norm_emb = stack_embed.pow(2).reduce(ee.Reducer.sum()).sqrt()

    # L2 norm of medoid vector (denominator term 2 — scalar constant)
    norm_med = ee.Number(sum(v ** 2 for v in medoid_vec)) \
                        .sqrt().max(ee.Number(1e-10))

    return dot.divide(norm_emb.max(ee.Number(1e-10))).divide(norm_med)

# Mosaic: for each pixel, show similarity to its own cluster's medoid
sim_layers = [
    cosine_sim_image(medoids_data['features'][i], embed_cols)
    .updateMask(aoi_cluster_img.eq(i))
    .rename('sim_to_nearest_medoid')
    for i in range(N_CLUSTERS)
]
sim_to_nearest = ee.ImageCollection(sim_layers).mosaic()

# 2-band output image: cluster_id (Int16) + similarity (Float32)
aoi_output_img = (
    aoi_cluster_img.toInt16()
    .addBands(sim_to_nearest.toFloat())
    .clip(aoi_geom)
)

print(f'  aoi_output_img bands: cluster_id (Int16), sim_to_nearest_medoid (Float32)')
print(f'  Ready for GeoTIFF export in Cell 7.')
print(f'\n✓ Cluster-then-Match complete.')

In [ ]:
# ============================================================================
# 7. MERGE ALL RESULTS + EXPORT OUTPUT FILES
# ============================================================================
# Outputs:
#
#   CorePoints_Covariates_BC_Canada.csv
#       Canonical 27-band stack at climate-filtered global core locations.
#       Column order enforced to exactly match the canonical band list in:
#         - GoogleEarthEngineAOICovariateAnalysis.js (AOI raster export, Step ⑥ band_names)
#         - CoastalBlueCarbon_LargeScaleCovariateExtraction.ipynb (BC Janousek cores)
#       Required for transfer learning in P4_05_Transfer_Learning_Modelling.R.
#
#   CorePoints_EmbeddingSimilarity_BC_Canada.csv
#       Cosine similarity to each of the k=5 AOI land-cover clusters.
#       embedding_max_sim used as per-core transfer-learning weight in R.
#
#   AOI_Medoids.csv / AOI_KMeans_Clusters.tif  (unchanged)
# ============================================================================

# ── Canonical band order ─────────────────────────────────────────
# This list MUST match the GEE JS script (band_names in Step ⑥)
# and CoastalBlueCarbon_LargeScaleCovariateExtraction.ipynb exactly.
CANONICAL_BANDS = [
    # Group 1 — Topography & Channels (7)
    'elevation_m', 'slope', 'elevationRelMHW', 'twi', 'dist_to_channel_m',
    'tidal_flat_prob', 'coastal_dist_m',
    # Group 2 — Sentinel-1 SAR (3)
    'VV_mean', 'VH_mean', 'VVVH_ratio',
    # Group 3 — Sentinel-2 Optical & Phenology (15: 9 raw + 6 derived)
    'B', 'G', 'R', 'B5', 'B6', 'B7', 'NIR', 'SWIR1', 'SWIR2',
    'NDVI_median', 'LSWI_median', 'mNDWI_median',
    'NDVI_stdDev', 'SAVI_median', 'tidal_wetness',
    # Group 4 — Climate (2)
    'MAT_C', 'MAP_mm',
]
assert len(CANONICAL_BANDS) == 27, f'Expected 27 canonical bands, got {len(CANONICAL_BANDS)}'
# Confirm removed bands are not in list
_removed_check = {'tpi', 'EVI_median', 'sg_soc_0_30cm', 'sg_soc_30_100cm', 'sg_soc_0_100cm'}
assert not _removed_check.intersection(CANONICAL_BANDS), \
    f'Removed bands found in CANONICAL_BANDS: {_removed_check.intersection(CANONICAL_BANDS)}'


# ── File 1: Covariates CSV (all 27 canonical bands) ──────────────
# TerraClimate values (MAT_C, MAP_mm) are already in df_filtered from Cell 5b.
# All other stacks extracted in Cell 5d are merged here.
final_cov = df_filtered.copy()
final_cov['profile_id'] = final_cov['profile_id'].astype(str)

final_cov = merge_df(final_cov, df_topo)
final_cov = merge_df(final_cov, df_s1)
final_cov = merge_df(final_cov, df_s2_raw)
final_cov = merge_df(final_cov, df_s2_derived)
final_cov = merge_df(final_cov, df_ndvi_std)   # NDVI_stdDev — phenology band

# Metadata columns to keep alongside the canonical covariates
meta_cols = ['profile_id', 'dataset', 'latitude', 'longitude']
meta_cols += [c for c in final_cov.columns
              if c not in meta_cols + CANONICAL_BANDS]

# Enforce canonical column order.
# Any canonical band missing (extraction failed) is filled with NaN and flagged.
for col in CANONICAL_BANDS:
    if col not in final_cov.columns:
        print(f'  ⚠ Canonical band missing (extraction failed): {col}')
        final_cov[col] = float('nan')

col_order = meta_cols + CANONICAL_BANDS
final_cov = final_cov[col_order]

print(f'CorePoints_Covariates_BC_Canada: {len(final_cov):,} rows × {len(final_cov.columns)} cols')
print(f'  → 27 canonical covariate bands confirmed')

# Coverage check across key bands
check_cols = [c for c in ['elevation_m', 'dist_to_channel_m', 'VV_mean',
                           'B5', 'NDVI_median', 'NDVI_stdDev', 'tidal_wetness', 'MAT_C']
              if c in final_cov.columns]
if check_cols:
    n_covered = final_cov[check_cols].notna().any(axis=1).sum()
    print(f'  Profiles with ≥1 covariate: {n_covered:,} / {len(final_cov):,} '
          f'({100*n_covered/len(final_cov):.1f}%)')
    print('\nNA rates by dataset (key covariates):')
    for col in check_cols:
        na_rates = final_cov.groupby('dataset')[col].apply(
            lambda x: x.isna().mean()).round(3)
        print(f'  {col}: {na_rates.to_dict()}')

# Safety check — confirm no removed bands leaked into output
removed_bands = {'tpi', 'EVI_median', 'NDBI_median', 'brightness', 'greenness',
                 'sg_soc_0_30cm', 'sg_soc_30_100cm', 'sg_soc_0_100cm'}
leaked = [c for c in removed_bands if c in final_cov.columns]
if leaked:
    print(f'\n  ⚠ WARNING: Removed bands still present — dropping: {leaked}')
    final_cov = final_cov.drop(columns=leaked)
else:
    print('\n  ✓ Removed bands (tpi, EVI, SoilGrids, NDBI, brightness, greenness) confirmed absent')

COV_OUTFILE = 'CorePoints_Covariates_BC_Canada.csv'
final_cov.to_csv(COV_OUTFILE, index=False)
files.download(COV_OUTFILE)
print(f'\n✓ Saved: {COV_OUTFILE}')


# ── File 2: Embedding Similarity CSV ─────────────────────────────
sim_df['profile_id'] = sim_df['profile_id'].astype(str)
sim_df_out = pd.merge(
    df_filtered[['profile_id', 'dataset', 'latitude', 'longitude']].assign(
        profile_id=lambda d: d['profile_id'].astype(str)
    ),
    sim_df, on='profile_id', how='left'
)
print(f'\nCorePoints_EmbeddingSimilarity_BC_Canada: '
      f'{len(sim_df_out):,} rows × {len(sim_df_out.columns)} cols')
sim_cols = [c for c in sim_df_out.columns if c.startswith('embedding_')]
print(f'  Similarity columns ({len(sim_cols)}): {sim_cols}')

SIM_OUTFILE = 'CorePoints_EmbeddingSimilarity_BC_Canada.csv'
sim_df_out.to_csv(SIM_OUTFILE, index=False)
files.download(SIM_OUTFILE)
print(f'✓ Saved: {SIM_OUTFILE}')


# ── File 3: AOI Medoid Vectors CSV ────────────────────────────────
medoid_rows = []
for feat in medoids_data['features']:
    row = {'cluster_id': feat['properties'].get('cluster_id')}
    row.update({b: feat['properties'].get(b, None) for b in embed_cols})
    medoid_rows.append(row)

df_medoids = pd.DataFrame(medoid_rows).sort_values('cluster_id').reset_index(drop=True)
MED_OUTFILE = 'AOI_Medoids.csv'
df_medoids.to_csv(MED_OUTFILE, index=False)
files.download(MED_OUTFILE)
print(f'\n✓ Saved: {MED_OUTFILE}  ({len(df_medoids)} cluster medoid vectors × {len(embed_cols)} bands)')


# ── File 4: AOI K-Means Cluster GeoTIFF → Google Drive ───────────
GEOTIFF_NAME = 'AOI_KMeans_Clusters'
export_task = ee.batch.Export.image.toDrive(
    image          = aoi_output_img,
    description    = GEOTIFF_NAME,
    folder         = 'GEE_Exports',
    fileNamePrefix = GEOTIFF_NAME,
    region         = aoi_geom,
    scale          = AOI_GRID_SCALE,
    crs            = 'EPSG:4326',
    maxPixels      = 1e9,
    fileFormat     = 'GeoTIFF'
)
export_task.start()
print(f'\n✓ GeoTIFF export task started: "{GEOTIFF_NAME}"')
print(f'  Bands: cluster_id (Int16), sim_to_nearest_medoid (Float32)')
print(f'  Monitor at: https://code.earthengine.google.com/  (Tasks tab)')


# ── Summary ───────────────────────────────────────────────────────
print('\n' + '═' * 65)
print('=== Extraction + Cluster-then-Match complete ===')
print(f'  {COV_OUTFILE}')
print(f'    → {len(final_cov):,} profiles × {len(final_cov.columns)} cols')
print(f'       (27 canonical covariates in canonical order)')
print(f'  {SIM_OUTFILE}')
print(f'    → {len(sim_df_out):,} profiles × {len(sim_df_out.columns)} cols')
print(f'       (use embedding_max_sim as transfer-learning weight in R)')
print(f'  {MED_OUTFILE}  |  {GEOTIFF_NAME}.tif (GDrive)')
print('═' * 65)
print()
print('── Canonical 27-band alignment check ────────────────────────')
print('  Band names MUST match JS script Step ⑥ band_names array')
print('  and CoastalBlueCarbon_LargeScaleCovariateExtraction.ipynb')
missing_check = [b for b in CANONICAL_BANDS if b not in final_cov.columns]
if missing_check:
    print(f'  ⚠ Missing canonical bands: {missing_check}')
else:
    print(f'  ✓ All 27 canonical bands present in output')
print(f'  Bands: {CANONICAL_BANDS}')


---
## Troubleshooting

### Batch timeouts / `EEException: Computation timed out`
The S2 extractor (`extract_batch_s2`) builds a fresh, spatially filtered median for each batch so GEE only processes ~20 local scenes. If timeouts still occur:
- Reduce `batch_size` from 25 → 10 in the `extract_batch_s2()` calls in Cell 5d
- Reduce `S2_IMAGE_LIMIT` from 20 → 10 in Cell 3
- Reduce `S2_BUFFER_M` from 5000 → 2000 in Cell 3
- For the Embedding extractor, try `batch_size=10` if 50 fails
- SoilGrids and TerraClimate are fast (large pixel, few scenes) — increase `batch_size` to 1000 if desired

### Why `extract_batch_s2()` instead of a pre-built mosaic
The previous approach pre-built a global S2 mosaic with `.limit(50)` and then called `reduceRegions()`. GEE ignores the image limit when computing a global mosaic — it still assembles the full collection graph before evaluating any pixels. The result was a timeout even at `batch_size=8`, because every call triggered computation over thousands of scenes globally.

`extract_batch_s2()` fixes this by calling `.filterBounds(region)` *before* the median, so GEE's lazy evaluation only ever materialises the ~20 scenes that actually overlap the batch bounding box.

### `EEException: User memory limit exceeded`
Remaining optimisations in place:
1. **Cell reorder** — TerraClimate extracted first (Cell 5), climate filter applied (Cell 5b), so S2 runs on ~40–53% fewer points (Cell 5d)
2. **S2 band split** — raw (6 bands) and derived (6 bands) extracted in separate calls
3. **Per-batch spatial filter** — S2 collection filtered to batch region before median
4. **tileScale=2** — splits computation tiles to reduce per-tile memory

### Climate filter removed too many / too few profiles
- Adjust `MAT_TOLERANCE` and `MAP_TOLERANCE` in Cell 3c
- Check `CorePoints_Climate_Filtered_OUT.csv` to see which profiles were removed
- Re-run from Cell 5b onwards (no need to re-extract TerraClimate)

### AOI centroid climate sample returns None
- Verify `AOI_ASSET` path in Cell 3c
- Fallback: hardcode `AOI_MAT = 4.5` and `AOI_MAP = 1100` (approximate coastal BC values)

### K-means / medoid extraction slow or fails (Cell 6)
- If `n_aoi_pixels` > 500k, increase `AOI_GRID_SCALE` to 60 m
- If `get_cluster_medoid()` times out, reduce `N_CLUSTERS` or `AOI_GRID_SCALE`
- `medoids_data` is plain Python — safe to save: `import json; json.dump(medoids_data, open('medoids_raw.json','w'))`

### GeoTIFF export to Google Drive fails (Cell 7)
- Check `export_task.status()` for `error_message`
- Try `scale=60` if the AOI is large
- The 3 CSV files are downloaded immediately regardless of GeoTIFF status

### Covariate alignment with other scripts
The 27 canonical band names in `CANONICAL_BANDS` (Cell 7) must match:
- `BlueCarbon_Covariate_Snapshot_25m_*.tif` band names produced by `GoogleEarthEngineAOICovariateAnalysis.js`
- Column names in `global_cores_with_gee_covariates_BC.csv` from `CoastalBlueCarbon_LargeScaleCovariateExtraction.ipynb`

If there is a mismatch, `P4_05_Transfer_Learning_Modelling.R` will fail with a column-not-found error.

### R model usage — embedding similarity weights
```r
# Run 1: Wadoux covariate-based weights
model_cov <- ranger(..., case.weights = df$wadoux_weight)

# Run 2: Embedding similarity weights
model_emb <- ranger(..., case.weights = df$embedding_max_sim)
```

### Expected output files
| File | Rows | Cols |
|------|------|------|
| `CorePoints_Covariates_BC_Canada.csv` | ~8,000–13,000 | meta cols + 27 canonical bands |
| `CorePoints_EmbeddingSimilarity_BC_Canada.csv` | ~8,000–13,000 | 4 meta + k+2 sim cols |
| `AOI_Medoids.csv` | k=5 | 65 (cluster_id + 64 bands) |
| `AOI_KMeans_Clusters.tif` | — | 2 bands (cluster_id, sim) |
| `CorePoints_Climate_Filtered_OUT.csv` | removed profiles | all metadata + MAT/MAP |
